In [242]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")

Available RAM: 1.0 GB


In [244]:

# ============================================================
# CELL 1: Imports & Configuration
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import warnings
import time
import math
from datetime import datetime

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
print("✔ Imports complete")

✔ Imports complete


In [245]:

# ============================================================
# CELL 2: Database Connection
# ============================================================

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"✔ Connected. Tables: {len(tables)}")

fights_count = pd.read_sql("SELECT COUNT(*) as c FROM fights_v2", conn)['c'].iloc[0]
print(f"✔ Total fights in DB: {fights_count}")


✔ Connected. Tables: 10
✔ Total fights in DB: 11127


In [246]:


# ============================================================
# CELL 3: Helper Functions
# ============================================================

def parse_reach(r):
    if pd.isna(r) or r == '--': return None
    try: return float(r.replace('"', ''))
    except: return None

def parse_height(h):
    if pd.isna(h) or h == '--': return None
    try:
        parts = h.replace('"', '').split("'")
        return int(parts[0]) * 12 + int(parts[1])
    except: return None

def parse_age(dob, fight_date):
    if pd.isna(dob) or pd.isna(fight_date): return None
    try:
        birth = datetime.strptime(dob, "%b %d, %Y")
        fight = datetime.strptime(fight_date, "%Y-%m-%d")
        return (fight - birth).days / 365.25
    except: return None

def parse_fight_duration(ending_round, ending_time):
    try:
        mins, secs = ending_time.split(':')
        final_round_minutes = int(mins) + int(secs) / 60
        return ((ending_round - 1) * 5) + final_round_minutes
    except: return 15.0

# Time decay: 1.5 year half-life (λ ≈ 0.00127)
LAM = math.log(2) / (1.5 * 365)

def time_decay_weights(dates, as_of_date, lam=LAM):
    as_of = datetime.strptime(as_of_date, "%Y-%m-%d")
    weights = []
    for d in dates:
        fight_dt = datetime.strptime(d, "%Y-%m-%d")
        days_ago = (as_of - fight_dt).days
        w = np.exp(-lam * max(days_ago, 0))
        weights.append(w)
    weights = np.array(weights)
    return weights / weights.sum() if weights.sum() > 0 else weights

def kish_effective_n(weights):
    """Kish effective sample size — more honest than raw count with decayed weights"""
    if weights.sum() == 0: return 0
    return (weights.sum() ** 2) / (weights ** 2).sum()
    
def normalize_weight_class(wc):
    if pd.isna(wc): return None
    wc = wc.strip()
    
    # Drop women's divisions
    if "Women's" in wc: return None
    
    # Map to base weight classes
    base_classes = [
        'Heavyweight', 'Light Heavyweight', 'Middleweight',
        'Welterweight', 'Lightweight', 'Featherweight',
        'Bantamweight', 'Flyweight', 'Catch Weight'
    ]
    for base in base_classes:
        if base in wc:
            return f'{base} Bout'
    
    return None

print(f"✔ Lambda (1.5yr half-life): {LAM:.6f}")
print("✔ Helper functions ready")

✔ Lambda (1.5yr half-life): 0.001266
✔ Helper functions ready


In [247]:
# ============================================================
# CELL 4: Pre-Cache Fight Data (with Poisson-Gamma & Beta-Binomial smoothing)
# ============================================================

print("Fetching all fight stats (no date filter)...")
start = time.time()

# ── Raw stats ──────────────────────────────────────────────
all_fight_stats_raw = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed)    as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed)         as td_landed,
        SUM(frs.td_attempted)      as td_attempted,
        SUM(frs.sub_attempts)      as sub_attempts,
        SUM(frs.ctrl_seconds)      as ctrl_seconds,
        SUM(frs.knockdowns)        as knockdowns
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr      ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = fr.fight_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id 
                                AND frs.fighter_id = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Actual fight duration
all_fight_stats_raw['actual_minutes'] = all_fight_stats_raw.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)
all_fight_stats_raw = all_fight_stats_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

# ── Step 1: Compute weight-class rate priors ─────────────────
print("Computing weight-class priors...")

all_fight_stats_raw['wc_norm'] = all_fight_stats_raw['weight_class'].apply(normalize_weight_class)
valid = all_fight_stats_raw[all_fight_stats_raw['wc_norm'].notna()].copy()

pg_stats = {
    'sig_str_landed': 0.7,
    'td_landed':      7.0,
    'sub_attempts':   12.0,
    'knockdowns':     20.0,
    'ctrl_seconds':   2.0,
}

wc_rates     = {}
global_rates = {}

for stat, tau in pg_stats.items():
    global_total       = valid[stat].sum()
    global_mins        = valid['actual_minutes'].sum()
    global_rates[stat] = global_total / global_mins if global_mins > 0 else 0

    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_rates:
            wc_rates[wc] = {}
        total = grp[stat].sum()
        mins  = grp['actual_minutes'].sum()
        wc_rates[wc][stat] = total / mins if mins > 0 else global_rates[stat]

bb_stats = {
    'str_acc': {'num': 'sig_str_landed', 'den': 'sig_str_attempted', 'tau': 0.7},
    'td_acc':  {'num': 'td_landed',      'den': 'td_attempted',      'tau': 7.0},
}

wc_bb_priors     = {}
global_bb_priors = {}

for stat, cfg in bb_stats.items():
    global_num = valid[cfg['num']].sum()
    global_den = valid[cfg['den']].sum()
    global_bb_priors[stat] = global_num / global_den if global_den > 0 else 0.5

    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_bb_priors:
            wc_bb_priors[wc] = {}
        num = grp[cfg['num']].sum()
        den = grp[cfg['den']].sum()
        wc_bb_priors[wc][stat] = num / den if den > 0 else global_bb_priors[stat]

print(f"✓ Weight class priors computed for {len(wc_rates)} classes")

# ── Step 2: Apply smoothing ────────────────────────────────
print("Applying Poisson-Gamma and Beta-Binomial smoothing...")

all_fight_stats = all_fight_stats_raw.copy()

def get_wc_rate(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_rates and stat in wc_rates[wc_norm]:
        return wc_rates[wc_norm][stat]
    return global_rates.get(stat, 0)

def get_wc_bb(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_bb_priors and stat in wc_bb_priors[wc_norm]:
        return wc_bb_priors[wc_norm][stat]
    return global_bb_priors.get(stat, 0.5)

for stat, tau in pg_stats.items():
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        t         = max(row['actual_minutes'], 0.1)
        wc_rate   = get_wc_rate(row['weight_class'], stat)
        x         = row[stat]
        post_rate = (wc_rate * tau + x) / (tau + t)
        smoothed.append(t * post_rate)
    all_fight_stats[f'{stat}_smooth'] = smoothed

for stat, cfg in bb_stats.items():
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        wc_rate = get_wc_bb(row['weight_class'], stat)
        tau     = cfg['tau']
        x       = row[cfg['num']]
        n       = row[cfg['den']]
        if n == 0:
            smoothed.append(wc_rate)
        else:
            smoothed.append((wc_rate * tau + x) / (tau + n))
    all_fight_stats[f'{stat}_smooth'] = smoothed

# ── Step 3: Compute per-minute rates from smoothed values ──
t = all_fight_stats['actual_minutes'].clip(lower=0.1)

all_fight_stats['slpm']              = all_fight_stats['sig_str_landed_smooth'] / t
all_fight_stats['str_acc']           = all_fight_stats['str_acc_smooth']
all_fight_stats['td_acc']            = all_fight_stats['td_acc_smooth']
all_fight_stats['td_avg']            = all_fight_stats['td_landed_smooth']    / (t / 15)
all_fight_stats['sub_avg']           = all_fight_stats['sub_attempts_smooth'] / (t / 15)
all_fight_stats['ctrl_time_per_min'] = all_fight_stats['ctrl_seconds_smooth'] / 60 / t
all_fight_stats['kd_per_min']        = all_fight_stats['knockdowns_smooth']   / t

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

# ── Step 4: Rebuild dicts ──────────────────────────────────
fighter_fights_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in all_fight_stats.groupby('fighter_id')
}

all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2 
        ON ff1.fight_id = ff2.fight_id 
       AND ff1.fighter_id != ff2.fighter_id
""", conn)

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Fighters cached: {len(fighter_fights_dict)}")
print(f"✓ Fight records:   {len(all_fight_stats)}")
print(f"✓ Opponent lookups:{len(opponents_dict)}")
print(f"✓ Smoothing: Poisson-Gamma for counts, Beta-Binomial for accuracies")

Fetching all fight stats (no date filter)...
Computing weight-class priors...
✓ Weight class priors computed for 8 classes
Applying Poisson-Gamma and Beta-Binomial smoothing...
Done in 10.9s
✓ Fighters cached: 3760
✓ Fight records:   21568
✓ Opponent lookups:22254
✓ Smoothing: Poisson-Gamma for counts, Beta-Binomial for accuracies


In [248]:
# ============================================================
# CELL 4b: Pre-Cache Strike Breakdown Data
# ============================================================

print("Fetching strike breakdown data...")
start = time.time()

strike_breakdown = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        ff.fighter_id,
        SUM(ss.head_landed)         as head_landed,
        SUM(ss.head_attempted)      as head_attempted,
        SUM(ss.body_landed)         as body_landed,
        SUM(ss.body_attempted)      as body_attempted,
        SUM(ss.leg_landed)          as leg_landed,
        SUM(ss.leg_attempted)       as leg_attempted,
        SUM(ss.distance_landed)     as distance_landed,
        SUM(ss.distance_attempted)  as distance_attempted,
        SUM(ss.clinch_landed)       as clinch_landed,
        SUM(ss.clinch_attempted)    as clinch_attempted,
        SUM(ss.ground_landed)       as ground_landed,
        SUM(ss.ground_attempted)    as ground_attempted
    FROM fight_round_sig_strikes_v2 ss
    JOIN fight_rounds_v2 fr      ON ss.round_id   = fr.round_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id
                                AND ss.fighter_id  = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Add fight duration
strike_breakdown['actual_minutes'] = strike_breakdown.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

# Per-minute rates and accuracy
for pos in ['head', 'body', 'leg', 'distance', 'clinch', 'ground']:
    strike_breakdown[f'{pos}_lpm'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown['actual_minutes']
    )
    strike_breakdown[f'{pos}_acc'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown[f'{pos}_attempted']
    )

strike_breakdown = strike_breakdown.replace([np.inf, -np.inf], np.nan).fillna(0)
strike_breakdown = strike_breakdown.sort_values('event_date').reset_index(drop=True)

# Dict keyed by fighter_id for O(1) lookups
strike_breakdown_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_breakdown.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Strike breakdown records: {len(strike_breakdown)}")
print(f"✓ Fighters with strike data: {len(strike_breakdown_dict)}")

Fetching strike breakdown data...
Done in 2.0s
✓ Strike breakdown records: 21568
✓ Fighters with strike data: 3760


In [249]:
# ============================================================
# CELL 4c: Pre-Cache Strike Defense (what opponents land on each fighter)
# ============================================================

print("Building strike defense lookup...")
start = time.time()

# For each fight, get opponent's strike breakdown against this fighter
strike_defense = strike_breakdown.copy()

# Map opponent's stats to the fighter they were landed on
opp_strike_rows = []
for _, row in strike_defense.iterrows():
    opp_id = opponents_dict.get((row['fight_id'], row['fighter_id']))
    if opp_id:
        opp_strike_rows.append({
            'fight_id':         row['fight_id'],
            'event_date':       row['event_date'],
            'fighter_id':       opp_id,          # the fighter who RECEIVED these strikes
            'head_allowed':     row['head_lpm'],
            'body_allowed':     row['body_lpm'],
            'leg_allowed':      row['leg_lpm'],
            'distance_allowed': row['distance_lpm'],
            'clinch_allowed':   row['clinch_lpm'],
            'ground_allowed':   row['ground_lpm'],
        })

strike_defense_df = pd.DataFrame(opp_strike_rows).sort_values('event_date').reset_index(drop=True)

# Dict keyed by fighter_id
strike_defense_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_defense_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Defense records: {len(strike_defense_df)}")
print(f"✔ Fighters with defense data: {len(strike_defense_dict)}")

Building strike defense lookup...
Done in 2.4s
✔ Defense records: 21568
✔ Fighters with defense data: 3761


In [250]:
# ============================================================
# CELL 4d: Pre-Cache Fighter Results
# ============================================================

print("Building fighter results cache...")

results_raw = pd.read_sql("""
    SELECT 
        ff.fighter_id,
        ff.fight_id,
        ff.result,
        f.method
    FROM fight_fighters_v2 ff
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
""", conn)

# Map (fighter_id, fight_id) → result
# Tag KO wins separately for ko_rate
fighter_results_dict = {}
for _, row in results_raw.iterrows():
    result = row['result']
    if result == 'win' and row['method'] == 'KO/TKO':
        tagged = 'ko_win'
    else:
        tagged = result
    fighter_results_dict[(row['fighter_id'], row['fight_id'])] = tagged

print(f"✔ Results cached: {len(fighter_results_dict)}")

Building fighter results cache...
✔ Results cached: 22254


In [251]:
# ============================================================
# CELL 4e: Pre-Cache Round 1 Stats
# ============================================================

print("Fetching Round 1 stats...")
start = time.time()

r1_stats = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.ending_round,
        f.ending_time,
        ff.fighter_id,
        frs.sig_str_landed    as r1_sig_str_landed,
        frs.sig_str_attempted as r1_sig_str_attempted,
        frs.ctrl_seconds      as r1_ctrl_seconds,
        frs.td_landed         as r1_td_landed,
        frs.td_attempted      as r1_td_attempted,
        frs.knockdowns        as r1_knockdowns,
        frs.reversals         as r1_reversals
    FROM fight_round_stats_v2 frs
    JOIN fights_v2 f          ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id    = ff.fight_id
                             AND frs.fighter_id = ff.fighter_id
    WHERE CAST(SUBSTR(frs.round_id, INSTR(frs.round_id, ':') + 1) AS INTEGER) = 1
""", conn)

# R1 exposure capped at 5 minutes
r1_stats['r1_minutes'] = r1_stats.apply(
    lambda r: min(parse_fight_duration(r['ending_round'], r['ending_time']), 5.0), axis=1
)

# Per-minute rates
r1_stats['r1_slpm']         = r1_stats['r1_sig_str_landed']  / r1_stats['r1_minutes']
r1_stats['r1_ctrl_per_min'] = r1_stats['r1_ctrl_seconds']    / 60 / r1_stats['r1_minutes']
r1_stats['r1_td_acc']       = r1_stats['r1_td_landed']       / r1_stats['r1_td_attempted']
r1_stats['r1_kd_per_min']   = r1_stats['r1_knockdowns']      / r1_stats['r1_minutes']
r1_stats['r1_rev_per_min']  = r1_stats['r1_reversals']       / r1_stats['r1_minutes']

r1_stats = r1_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
r1_stats = r1_stats.sort_values('event_date').reset_index(drop=True)

r1_stats_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in r1_stats.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ R1 records: {len(r1_stats)}")
print(f"✔ Fighters with R1 data: {len(r1_stats_dict)}")

Fetching Round 1 stats...
Done in 1.2s
✔ R1 records: 21568
✔ Fighters with R1 data: 3760


In [252]:
# ============================================================
# CELL 4f: Weight Class Priors
# ============================================================

print("Building weight class priors...")

STATS_FOR_PRIORS = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
                    'ctrl_time_per_min', 'kd_per_min']

# Add normalized weight class to all_fight_stats
all_fight_stats['weight_class_norm'] = all_fight_stats['weight_class'].apply(normalize_weight_class)

# Filter to valid weight classes only
valid_stats = all_fight_stats[all_fight_stats['weight_class_norm'].notna()].copy()

# Compute per-weight-class median and MAD for each stat
wc_priors = {}
global_priors = {}

for stat in STATS_FOR_PRIORS:
    # Global prior as fallback
    global_median = float(np.median(valid_stats[stat].values))
    global_mad    = float(np.median(np.abs(valid_stats[stat].values - global_median)))
    global_priors[stat] = {'mean': global_median, 'mad': max(global_mad, 0.01)}

    # Per weight class
    for wc, grp in valid_stats.groupby('weight_class_norm'):
        if wc not in wc_priors:
            wc_priors[wc] = {}
        vals   = grp[stat].values
        median = float(np.median(vals))
        mad    = float(np.median(np.abs(vals - median)))
        wc_priors[wc][stat] = {'mean': median, 'mad': max(mad, 0.01)}

print(f"✔ Weight classes with priors: {len(wc_priors)}")
print(f"✔ Global fallback priors: {len(global_priors)} stats")
print("\nWeight classes:")
for wc in sorted(wc_priors.keys()):
    print(f"  {wc}")

Building weight class priors...
✔ Weight classes with priors: 8
✔ Global fallback priors: 7 stats

Weight classes:
  Bantamweight Bout
  Catch Weight Bout
  Featherweight Bout
  Flyweight Bout
  Heavyweight Bout
  Lightweight Bout
  Middleweight Bout
  Welterweight Bout


In [253]:
# ============================================================
# CELL 4g: Pre-Cache Per-Fight AdjPerf Z-Scores
# ============================================================
# NEW: After computing each fight's AdjPerf snapshot, store it
# so we can decay-average historical z-scores in Cell 6.
#
# WARNING: This cell is slow (~10-15 min) since it computes
# AdjPerf for every fighter across every fight historically.
# Run once and reuse fighter_adjperf_history dict.
#
# This resolves the key finding from the author's videos:
#   His naming: sig_str_land_ratio_dec_adjperf_dec_avg_diff
#   Means:      AdjPerf z-scores are ALSO decayed averaged
#               across recent fights, not just used as snapshots.
#
# Pipeline comparison:
#   Yours (before): dec_avg → AdjPerf snapshot → diff
#   His (confirmed): dec_avg → AdjPerf snapshot → decay avg → diff

STATS = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
         'ctrl_time_per_min', 'kd_per_min']

print("Pre-caching per-fight AdjPerf z-scores...")
start = time.time()

# Dict to store: {fighter_id: DataFrame with columns
#   [fight_id, event_date, {stat}_adjperf_snapshot]}
fighter_adjperf_history = {}

# We need to compute AdjPerf for every fight in all_fight_stats
# using only prior fight history — strict no-leakage
all_fights_sorted = all_fight_stats.sort_values('event_date').reset_index(drop=True)

pop_means = {s: float(np.median(all_fight_stats[s].values)) for s in STATS}
pop_mads  = {s: float(np.median(np.abs(
    all_fight_stats[s].values - pop_means[s]
))) for s in STATS}

adjperf_rows = []

for _, fight_row in all_fights_sorted.iterrows():
    fid   = fight_row['fighter_id']
    fdate = fight_row['event_date']
    fid_fight = fight_row['fight_id']

    # Get opponent
    opp_id = opponents_dict.get((fid_fight, fid))
    if not opp_id or opp_id not in fighter_fights_dict:
        continue

    fighter_hist = fighter_fights_dict[fid]
    opp_hist     = fighter_fights_dict[opp_id]

    fighter_prev = fighter_hist[fighter_hist['event_date'] < fdate].tail(WINDOW)
    opp_prev     = opp_hist[opp_hist['event_date'] < fdate]

    if len(fighter_prev) == 0:
        continue

    row = {'fighter_id': fid, 'fight_id': fid_fight, 'event_date': fdate}

    f_weights = time_decay_weights(fighter_prev['event_date'].tolist(), fdate)
    f_n_eff   = kish_effective_n(f_weights)

    for stat in STATS:
        # Layer 1: Fighter decayed avg
        dec_avg  = np.average(fighter_prev[stat].values, weights=f_weights)
        smoothed = bayesian_smooth(dec_avg, f_n_eff, pop_means[stat], K_MEAN)

        # Layer 2/3: Opponent allowed baseline → AdjPerf snapshot
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opp_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_hist  = fighter_fights_dict[opp_opp_id]
                opp_opp_this  = opp_opp_hist[
                    opp_opp_hist['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    opp_allowed.append(opp_opp_this[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])

        if len(opp_allowed) >= 2:
            opp_w     = time_decay_weights(opp_dates, fdate)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mu    = bayesian_smooth(
                np.average(opp_allowed, weights=opp_w),
                opp_n_eff, pop_means[stat], K_MEAN
            )
            opp_sigma = max(bayesian_smooth(
                np.average(np.abs(np.array(opp_allowed) -
                    np.average(opp_allowed, weights=opp_w)), weights=opp_w),
                opp_n_eff, pop_mads[stat], K_MAD
            ), 0.01)
            snapshot = float(np.clip((smoothed - opp_mu) / opp_sigma, -7, 7))
        else:
            snapshot = 0.0

        row[f'{stat}_adjperf_snapshot'] = snapshot

    adjperf_rows.append(row)

adjperf_history_df = pd.DataFrame(adjperf_rows).sort_values('event_date').reset_index(drop=True)

# Build per-fighter dict for O(1) lookups
fighter_adjperf_history = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in adjperf_history_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ AdjPerf history cached for {len(fighter_adjperf_history)} fighters")
print(f"✓ Total records: {len(adjperf_history_df)}")

Pre-caching per-fight AdjPerf z-scores...
Done in 304.4s
✓ AdjPerf history cached for 2661 fighters
✓ Total records: 17755


In [254]:
# Confirm round number is the suffix after the colon
pd.read_sql("""
    SELECT 
        round_id,
        CAST(SUBSTR(round_id, INSTR(round_id, ':') + 1) AS INTEGER) as round_num,
        COUNT(*) as rows
    FROM fight_round_stats_v2
    GROUP BY round_num
    ORDER BY round_num
""", conn)

,round_id,round_num,rows
0,0005e00b07cee542:1,1,21568
1,0005e00b07cee542:2,2,14686
2,0005e00b07cee542:3,3,10946
3,0005e00b07cee542:4,4,840
4,0005e00b07cee542:5,5,728
5,7f0e5ac5bcb5a0f3:6,6,4


In [255]:

# ============================================================
# CELL 5: Generate Base Fights (2016–2026, clean methods only)
# ============================================================

EXCLUDED_METHODS = ['S-DEC', 'M-DEC', 'Overturned', 'CNC', 'DQ', 'Other', 'Decision']
MIN_PREV_FIGHTS  = 3   # both fighters need this many prior UFC fights

def generate_base_fights_filtered(start_date='2016-01-01', end_date='2026-12-31'):
    excl = "', '".join(EXCLUDED_METHODS)
    fights = pd.read_sql(f"""
        SELECT 
            f.fight_id,
            f.event_date,
            f.event_name,
            f.weight_class,
            f.method,
            f.ending_round,
            f.ending_time,
            ff1.fighter_id  as fighter_1_id,
            fv1.name        as fighter_1_name,
            ff2.fighter_id  as fighter_2_id,
            fv2.name        as fighter_2_name,
            CASE WHEN ff1.result = 'win' THEN 1 ELSE 0 END as winner
        FROM fights_v2 f
        JOIN fight_fighters_v2 ff1 ON f.fight_id = ff1.fight_id AND ff1.corner = 'fighter_1'
        JOIN fight_fighters_v2 ff2 ON f.fight_id = ff2.fight_id AND ff2.corner = 'fighter_2'
        JOIN fighters_v2 fv1       ON ff1.fighter_id = fv1.fighter_id
        JOIN fighters_v2 fv2       ON ff2.fighter_id = fv2.fighter_id
        WHERE f.event_date >= '{start_date}'
          AND f.event_date <= '{end_date}'
          AND f.method NOT IN ('{excl}')
        ORDER BY f.event_date
    """, conn)

    valid = []
    for _, fight in fights.iterrows():
        f1_id, f2_id, fdate = fight['fighter_1_id'], fight['fighter_2_id'], fight['event_date']

        f1_prev = len(fighter_fights_dict[f1_id][fighter_fights_dict[f1_id]['event_date'] < fdate]) \
                  if f1_id in fighter_fights_dict else 0
        f2_prev = len(fighter_fights_dict[f2_id][fighter_fights_dict[f2_id]['event_date'] < fdate]) \
                  if f2_id in fighter_fights_dict else 0

        if f1_prev >= MIN_PREV_FIGHTS and f2_prev >= MIN_PREV_FIGHTS:
            valid.append(fight)

    filtered = pd.DataFrame(valid)
    print(f"Raw fights  (2016-{end_date[:4]}): {len(fights)}")
    print(f"After {MIN_PREV_FIGHTS}+ fight filter: {len(filtered)}")
    print(f"Dropped: {len(fights)-len(filtered)} ({(len(fights)-len(filtered))/len(fights)*100:.1f}%)")
    return filtered


base_fights = generate_base_fights_filtered()
print(f"\n✔ Base fights shape: {base_fights.shape}")
print(f"✔ Date range: {base_fights['event_date'].min()} to {base_fights['event_date'].max()}")
print(f"✔ Winner distribution: {base_fights['winner'].value_counts().to_dict()}")

Raw fights  (2016-2026): 4878
After 3+ fight filter: 2474
Dropped: 2404 (49.3%)

✔ Base fights shape: (2474, 12)
✔ Date range: 2016-01-02 to 2026-02-07
✔ Winner distribution: {0: 1260, 1: 1214}


In [256]:
# ============================================================
# CELL 6: Three-Layer Feature Function (with Kish n_eff + split K)
# ============================================================

# Reverted to 65.8% baseline — str_acc back in STATS
STATS  = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
          'ctrl_time_per_min', 'kd_per_min']
K_MEAN = 4.0
K_MAD  = 6.0
WINDOW = 5

def bayesian_smooth(observed, n_eff, population_mean, k):
    w = n_eff / (n_eff + k)
    return w * observed + (1 - w) * population_mean

def calculate_three_layer_features_v2(fighter_id, opponent_id, as_of_date,
                                       stats=STATS, window=WINDOW):
    if fighter_id not in fighter_fights_dict or opponent_id not in fighter_fights_dict:
        return None

    fighter_hist  = fighter_fights_dict[fighter_id]
    opponent_hist = fighter_fights_dict[opponent_id]

    fighter_prev  = fighter_hist[fighter_hist['event_date']   < as_of_date]
    opponent_prev = opponent_hist[opponent_hist['event_date'] < as_of_date]

    if len(fighter_prev) == 0 or len(opponent_prev) == 0:
        return None

    pop_means = {s: float(np.median(all_fight_stats[s].values)) for s in stats}
    pop_mads  = {s: float(np.median(np.abs(
        all_fight_stats[s].values - pop_means[s]
    ))) for s in stats}

    fighter_recent  = fighter_prev.tail(window)
    fighter_weights = time_decay_weights(fighter_recent['event_date'].tolist(), as_of_date)
    fighter_n_eff   = kish_effective_n(fighter_weights)

    features = {}

    for stat in stats:
        # --- Layer 1: Fighter's time-decayed average ---
        decayed_avg   = np.average(fighter_recent[stat].values, weights=fighter_weights)
        smoothed_stat = bayesian_smooth(decayed_avg, fighter_n_eff, pop_means[stat], K_MEAN)
        features[f'{stat}_dec_avg'] = smoothed_stat

        # --- Layer 2: Opponent's allowed baseline ---
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opponent_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_fights     = fighter_fights_dict[opp_opp_id]
                opp_opp_this_fight = opp_opp_fights[
                    opp_opp_fights['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this_fight) > 0:
                    opp_allowed.append(opp_opp_this_fight[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])

        if len(opp_allowed) >= 2:
            opp_weights = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff   = kish_effective_n(opp_weights)
            opp_median  = np.average(opp_allowed, weights=opp_weights)
            abs_devs    = np.abs(np.array(opp_allowed) - opp_median)
            opp_mad     = np.average(abs_devs, weights=opp_weights)

            opp_mu    = bayesian_smooth(opp_median, opp_n_eff, pop_means[stat], K_MEAN)
            opp_sigma = max(
                bayesian_smooth(opp_mad, opp_n_eff, pop_mads[stat], K_MAD),
                0.01
            )
            features[f'{stat}_opp_dec_avg'] = opp_mu
            features[f'{stat}_opp_mad']     = opp_sigma
            features[f'{stat}_adjperf']     = np.clip(
                (smoothed_stat - opp_mu) / opp_sigma, -7, 7
            )
        else:
            features[f'{stat}_opp_dec_avg'] = pop_means[stat]
            features[f'{stat}_opp_mad']     = 1.0
            features[f'{stat}_adjperf']     = 0.0

        # --- dec_adjperf — decay average of historical z-scores ---
        if fighter_id in fighter_adjperf_history:
            ap_hist  = fighter_adjperf_history[fighter_id]
            ap_prev  = ap_hist[ap_hist['event_date'] < as_of_date].tail(window)
            snap_col = f'{stat}_adjperf_snapshot'

            if len(ap_prev) > 0 and snap_col in ap_prev.columns:
                ap_weights = time_decay_weights(
                    ap_prev['event_date'].tolist(), as_of_date
                )
                ap_n_eff   = kish_effective_n(ap_weights)
                ap_dec_avg = np.average(ap_prev[snap_col].values, weights=ap_weights)
                features[f'{stat}_dec_adjperf'] = bayesian_smooth(
                    ap_dec_avg, ap_n_eff, 0.0, K_MEAN
                )
            else:
                features[f'{stat}_dec_adjperf'] = 0.0
        else:
            features[f'{stat}_dec_adjperf'] = 0.0

    return features


print("✓ Three-layer feature function v2 ready (reverted to 65.8% baseline)")
print(f"  STATS: {STATS}")
print(f"  K_MEAN={K_MEAN}, K_MAD={K_MAD}, WINDOW={WINDOW}")
print(f"  Features per stat: dec_avg, opp_dec_avg, opp_mad, adjperf, dec_adjperf")

✓ Three-layer feature function v2 ready (reverted to 65.8% baseline)
  STATS: ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min']
  K_MEAN=4.0, K_MAD=6.0, WINDOW=5
  Features per stat: dec_avg, opp_dec_avg, opp_mad, adjperf, dec_adjperf


In [257]:
# ============================================================
# CELL 6b: Strike Breakdown Feature Function
# ============================================================

# Reverted to 65.8% baseline — head_acc back in STRIKE_STATS_OFF
STRIKE_STATS_OFF = ['head_lpm', 'body_lpm', 'leg_lpm',
                    'distance_lpm', 'clinch_lpm', 'ground_lpm',
                    'head_acc', 'body_acc', 'distance_acc']

STRIKE_STATS_DEF = ['head_allowed', 'body_allowed', 'leg_allowed',
                    'distance_allowed', 'clinch_allowed', 'ground_allowed']

STRIKE_ADJPERF_OFF = STRIKE_STATS_OFF
STRIKE_ADJPERF_DEF = STRIKE_STATS_DEF


def calculate_strike_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    # Precompute population priors
    off_priors = {
        s: {
            'mean': float(strike_breakdown[s].median()),
            'mad':  float(max(np.median(np.abs(
                strike_breakdown[s].values - strike_breakdown[s].median()
            )), 0.01))
        } for s in STRIKE_STATS_OFF
    }
    def_priors = {
        s: {
            'mean': float(strike_defense_df[s].median()),
            'mad':  float(max(np.median(np.abs(
                strike_defense_df[s].values - strike_defense_df[s].median()
            )), 0.01))
        } for s in STRIKE_STATS_DEF
    }

    # --- Fighter's own offensive decayed averages ---
    fighter_off_smoothed = {}
    if fighter_id in strike_breakdown_dict:
        hist = strike_breakdown_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_OFF:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = 0.0
                    fighter_off_smoothed[s]  = 0.0
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, off_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_off_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_OFF:
                features[f'{s}_dec_avg'] = 0.0
                fighter_off_smoothed[s]  = 0.0
    else:
        for s in STRIKE_STATS_OFF:
            features[f'{s}_dec_avg'] = 0.0
            fighter_off_smoothed[s]  = 0.0

    # --- Fighter's own defensive decayed averages ---
    fighter_def_smoothed = {}
    if fighter_id in strike_defense_dict:
        hist = strike_defense_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_DEF:
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, def_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_def_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_DEF:
                features[f'{s}_dec_avg'] = 0.0
                fighter_def_smoothed[s]  = 0.0
    else:
        for s in STRIKE_STATS_DEF:
            features[f'{s}_dec_avg'] = 0.0
            fighter_def_smoothed[s]  = 0.0

    # --- AdjPerf for all offensive stats ---
    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in STRIKE_ADJPERF_OFF:
            observed = fighter_off_smoothed[s]
            pop_mean = off_priors[s]['mean']
            pop_mad  = off_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                    opp_opp_hist = strike_breakdown_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = np.average(
                    np.abs(np.array(opp_allowed) - opp_mean), weights=opp_w
                )
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0

    # --- AdjPerf for all defensive stats ---
    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in STRIKE_ADJPERF_DEF:
            observed = fighter_def_smoothed[s]
            pop_mean = def_priors[s]['mean']
            pop_mad  = def_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_defense_dict:
                    opp_opp_hist = strike_defense_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = np.average(
                    np.abs(np.array(opp_allowed) - opp_mean), weights=opp_w
                )
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0

    return features


print("✓ Strike feature function ready (reverted to 65.8% baseline)")
print(f"  Offensive: {len(STRIKE_STATS_OFF)} stats × 2 (dec_avg + adjperf)")
print(f"  head_acc restored")
print(f"  Defensive: {len(STRIKE_STATS_DEF)} stats × 2 (dec_avg + adjperf)")

✓ Strike feature function ready (reverted to 65.8% baseline)
  Offensive: 9 stats × 2 (dec_avg + adjperf)
  head_acc restored
  Defensive: 6 stats × 2 (dec_avg + adjperf)


In [258]:
# ============================================================
# CELL 6c: Rolling Career Stats (updated)
# ============================================================

def calculate_career_stats(fighter_id, opponent_id, fight_id, as_of_date, window=10):
    features = {}

    defaults = {
        'days_since_last_fight': 180,
        'win_ratio':             0.5,
        'win_adjperf':           0.0,
        'ko_rate':               0.0,
        'ko_opp_dec_avg':        0.0,
        'td_defense':            0.5,
        'td_land_ratio_opp':     0.0,
        'ctrl_ratio_opp':        0.0,
        'sub_att_allowed_pm':    0.0,
        'kd_allowed_pm':         0.0,
    }

    if fighter_id not in fighter_fights_dict:
        return defaults

    hist = fighter_fights_dict[fighter_id]
    prev = hist[hist['event_date'] < as_of_date]

    if len(prev) == 0:
        return defaults

    # --- Days since last fight ---
    last_date = prev.iloc[-1]['event_date']
    features['days_since_last_fight'] = (
        datetime.strptime(as_of_date, "%Y-%m-%d") -
        datetime.strptime(last_date,  "%Y-%m-%d")
    ).days

    # --- Win ratio + KO rate ---
    prior_fids = prev['fight_id'].tolist()
    results    = [fighter_results_dict.get((fighter_id, fid)) for fid in prior_fids]
    results    = [r for r in results if r is not None]

    if len(results) > 0:
        win_ratio = sum(1 for r in results if r == 'win') / len(results)
        ko_rate   = sum(1 for r in results if r == 'ko_win') / len(results)
        features['win_ratio'] = win_ratio
        features['ko_rate']   = ko_rate
    else:
        win_ratio = 0.5
        ko_rate   = 0.0
        features['win_ratio'] = win_ratio
        features['ko_rate']   = ko_rate

    # --- Win rate AdjPerf ---
    # Compare fighter's win rate vs what opponents typically allow
    if opponent_id in fighter_fights_dict:
        opp_hist = fighter_fights_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        opp_win_rates, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id:
                opp_opp_fids = fighter_fights_dict.get(opp_opp_id)
                if opp_opp_fids is not None:
                    opp_opp_prev = opp_opp_fids[
                        opp_opp_fids['event_date'] < opp_fight['event_date']
                    ]
                    opp_opp_results = [
                        fighter_results_dict.get((opp_opp_id, fid))
                        for fid in opp_opp_prev['fight_id'].tolist()
                    ]
                    opp_opp_results = [r for r in opp_opp_results if r is not None]
                    if len(opp_opp_results) > 0:
                        opp_win_rates.append(
                            sum(1 for r in opp_opp_results if r == 'win') / len(opp_opp_results)
                        )
                        opp_dates.append(opp_fight['event_date'])

        if len(opp_win_rates) >= 2:
            opp_w     = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mean  = np.average(opp_win_rates, weights=opp_w)
            opp_mad   = max(np.average(
                np.abs(np.array(opp_win_rates) - opp_mean), weights=opp_w
            ), 0.01)
            pop_mean  = 0.5
            pop_mad   = 0.15
            mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
            sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
            features['win_adjperf'] = np.clip((win_ratio - mu) / sigma, -7, 7)
        else:
            features['win_adjperf'] = 0.0
    else:
        features['win_adjperf'] = 0.0

    # --- Opponent stats (rolling window) ---
    recent = prev.tail(window)
    total_td_att, total_td_land   = 0, 0
    total_sub_att, total_minutes  = 0, 0
    total_kd, total_minutes_kd    = 0, 0
    total_ctrl, total_fight_mins  = 0, 0
    opp_ko_rates, opp_ko_dates    = [], []
    opp_td_ratios, opp_td_dates   = [], []

    for _, fight_row in recent.iterrows():
        opp_id = opponents_dict.get((fight_row['fight_id'], fighter_id))
        if opp_id and opp_id in fighter_fights_dict:
            opp_hist  = fighter_fights_dict[opp_id]
            opp_this  = opp_hist[opp_hist['fight_id'] == fight_row['fight_id']]
            if len(opp_this) > 0:
                opp_row = opp_this.iloc[0]

                # TD defense
                total_td_att  += opp_row['td_attempted']
                total_td_land += opp_row['td_landed']

                # Sub attempts allowed per minute
                total_sub_att += opp_row['sub_attempts']
                total_minutes += opp_row['actual_minutes']

                # KD allowed per minute
                total_kd         += opp_row['kd_per_min'] * opp_row['actual_minutes']
                total_minutes_kd += opp_row['actual_minutes']

                # Control ratio opponent allowed
                total_ctrl       += opp_row['ctrl_time_per_min'] * opp_row['actual_minutes']
                total_fight_mins += opp_row['actual_minutes']

                # KO rate opponent allowed
                opp_prev_fights = opp_hist[opp_hist['event_date'] < fight_row['event_date']]
                opp_results = [
                    fighter_results_dict.get((opp_id, fid))
                    for fid in opp_prev_fights['fight_id'].tolist()
                ]
                opp_results = [r for r in opp_results if r is not None]
                if len(opp_results) > 0:
                    opp_ko_rates.append(
                        sum(1 for r in opp_results if r == 'ko_win') / len(opp_results)
                    )
                    opp_ko_dates.append(fight_row['event_date'])

                # TD landing ratio opponent allowed
                if opp_row['td_attempted'] > 0:
                    opp_td_ratios.append(opp_row['td_landed'] / opp_row['td_attempted'])
                    opp_td_dates.append(fight_row['event_date'])

    features['td_defense'] = (
        1 - (total_td_land / total_td_att) if total_td_att > 0 else 0.5
    )
    features['sub_att_allowed_pm'] = (
        total_sub_att / total_minutes if total_minutes > 0 else 0.0
    )
    features['kd_allowed_pm'] = (
        total_kd / total_minutes_kd if total_minutes_kd > 0 else 0.0
    )
    features['ctrl_ratio_opp'] = (
        total_ctrl / total_fight_mins if total_fight_mins > 0 else 0.0
    )

    # KO rate opponent allowed — decayed average
    if len(opp_ko_rates) >= 2:
        opp_w = time_decay_weights(opp_ko_dates, as_of_date)
        features['ko_opp_dec_avg'] = float(np.average(opp_ko_rates, weights=opp_w))
    else:
        features['ko_opp_dec_avg'] = 0.0

    # TD land ratio opponent allowed — decayed average
    if len(opp_td_ratios) >= 2:
        opp_w = time_decay_weights(opp_td_dates, as_of_date)
        features['td_land_ratio_opp'] = float(np.average(opp_td_ratios, weights=opp_w))
    else:
        features['td_land_ratio_opp'] = 0.0

    return features


print("✔ Career stats function ready (updated)")

✔ Career stats function ready (updated)


In [259]:
# ============================================================
# CELL 6d: Round 1 Feature Function (updated)
# ============================================================

R1_STATS = ['r1_slpm', 'r1_ctrl_per_min', 'r1_td_acc',
            'r1_kd_per_min', 'r1_rev_per_min', 'r1_td_att_per_min']

def calculate_r1_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    # Precompute population priors
    r1_priors = {}
    for s in R1_STATS:
        if s in r1_stats.columns:
            median = float(r1_stats[s].median())
            mad    = float(max(np.median(np.abs(r1_stats[s].values - median)), 0.01))
            r1_priors[s] = {'mean': median, 'mad': mad}

    # Need td_att per minute from r1_stats
    if 'r1_td_att_per_min' not in r1_stats.columns:
        r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_attempted'] / r1_stats['r1_minutes']
        r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_att_per_min'].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(0)
        # Update dict
        for fid in r1_stats_dict:
            r1_stats_dict[fid] = r1_stats[r1_stats['fighter_id'] == fid].sort_values(
                'event_date'
            ).reset_index(drop=True)
        # Recompute prior
        median = float(r1_stats['r1_td_att_per_min'].median())
        mad    = float(max(np.median(np.abs(
            r1_stats['r1_td_att_per_min'].values - median
        )), 0.01))
        r1_priors['r1_td_att_per_min'] = {'mean': median, 'mad': mad}

    # --- Fighter's own R1 decayed averages ---
    fighter_smoothed = {}
    if fighter_id in r1_stats_dict:
        hist = r1_stats_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in R1_STATS:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = 0.0
                    fighter_smoothed[s]      = 0.0
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, r1_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_smoothed[s]      = smoothed
        else:
            for s in R1_STATS:
                features[f'{s}_dec_avg'] = 0.0
                fighter_smoothed[s]      = 0.0
    else:
        for s in R1_STATS:
            features[f'{s}_dec_avg'] = 0.0
            fighter_smoothed[s]      = 0.0

    # --- Full AdjPerf + opponent allowed for all R1 stats ---
    if opponent_id in r1_stats_dict:
        opp_hist = r1_stats_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in R1_STATS:
            observed = fighter_smoothed[s]
            pop_mean = r1_priors[s]['mean']
            pop_mad  = r1_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in r1_stats_dict:
                    opp_opp_hist = r1_stats_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = np.average(
                    np.abs(np.array(opp_allowed) - opp_mean), weights=opp_w
                )
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf']     = np.clip((observed - mu) / sigma, -7, 7)
                features[f'{s}_opp_dec_avg'] = mu
            else:
                features[f'{s}_adjperf']     = 0.0
                features[f'{s}_opp_dec_avg'] = pop_mean

    # --- Leg landing R1 opponent allowed ---
    # Uses strike_breakdown_dict filtered to R1 data
    leg_opp_allowed, leg_opp_dates = [], []
    if opponent_id in r1_stats_dict:
        opp_prev = r1_stats_dict[opponent_id]
        opp_prev = opp_prev[opp_prev['event_date'] < as_of_date]
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                opp_opp_strikes = strike_breakdown_dict[opp_opp_id]
                opp_opp_this    = opp_opp_strikes[
                    opp_opp_strikes['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    leg_opp_allowed.append(opp_opp_this['leg_lpm'].iloc[0])
                    leg_opp_dates.append(opp_fight['event_date'])

    if len(leg_opp_allowed) >= 2:
        opp_w = time_decay_weights(leg_opp_dates, as_of_date)
        features['leg_land_r1_opp_dec_avg'] = float(np.average(leg_opp_allowed, weights=opp_w))
    else:
        features['leg_land_r1_opp_dec_avg'] = 0.0

    return features


print("✔ R1 feature function ready (updated)")
print(f"  R1 stats: {len(R1_STATS)} — each gets dec_avg + adjperf + opp_dec_avg")
print(f"  Plus: leg_land_r1_opp_dec_avg")

✔ R1 feature function ready (updated)
  R1 stats: 6 — each gets dec_avg + adjperf + opp_dec_avg
  Plus: leg_land_r1_opp_dec_avg


In [261]:
# ============================================================
# CELL 7: Generate Features for All Fights
# ============================================================

def generate_features(base_df):
    rows  = []
    total = len(base_df)

    for idx, fight in base_df.iterrows():
        if idx % 500 == 0:
            print(f"  Processing {idx}/{total}...")

        f1_id = fight['fighter_1_id']
        f2_id = fight['fighter_2_id']
        fdate = fight['event_date']
        fid   = fight['fight_id']


        # Replace with:
        f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, fdate)
        f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, fdate)

        if f1_feats is None or f2_feats is None:
            continue

        # --- Strike breakdown features ---
        f1_feats.update(calculate_strike_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_strike_features(f2_id, f1_id, fdate))

        # --- Round 1 features ---
        f1_feats.update(calculate_r1_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_r1_features(f2_id, f1_id, fdate))

        # --- Rolling career stats ---
        f1_feats.update(calculate_career_stats(f1_id, f2_id, fid, fdate))
        f2_feats.update(calculate_career_stats(f2_id, f1_id, fid, fdate))
        # --- Assemble row ---
        row = {'fight_id': fid}
        for k, v in f1_feats.items(): row[f'f1_{k}'] = v
        for k, v in f2_feats.items(): row[f'f2_{k}'] = v
        for k   in f1_feats.keys():   row[f'diff_{k}'] = row[f'f1_{k}'] - row[f'f2_{k}']
        rows.append(row)

    feats_df = pd.DataFrame(rows)
    return base_df.merge(feats_df, on='fight_id', how='inner')


print("Generating three-layer features...")
start = time.time()
fight_features = generate_features(base_fights)
print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Shape: {fight_features.shape}")
print(f"✔ Features per fighter: {len([c for c in fight_features.columns if c.startswith('f1_')])}")

Generating three-layer features...
  Processing 1000/2474...
  Processing 1500/2474...
  Processing 4000/2474...
  Processing 4500/2474...
Done in 650.3s
✔ Shape: (2474, 294)
✔ Features per fighter: 94


In [262]:
# ============================================================
# CELL 8: Physical Features & Experience
# ============================================================

def add_physical_and_experience(df, conn):
    all_ids  = set(df['fighter_1_id'].unique()) | set(df['fighter_2_id'].unique())
    fighters = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob
        FROM fighters_v2
        WHERE fighter_id IN ({','.join(f"'{fid}'" for fid in all_ids)})
    """, conn)

    fighters['reach_in']  = fighters['reach'].apply(parse_reach)
    fighters['height_in'] = fighters['height'].apply(parse_height)

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        df = df.merge(
            fighters[['fighter_id', 'reach_in', 'height_in', 'dob']],
            left_on=fid_col, right_on='fighter_id', how='left'
        ).drop('fighter_id', axis=1).rename(columns={
            'reach_in': f'{prefix}_reach', 'height_in': f'{prefix}_height', 'dob': f'{prefix}_dob'
        })

    df['f1_age'] = df.apply(lambda r: parse_age(r['f1_dob'], r['event_date']), axis=1)
    df['f2_age'] = df.apply(lambda r: parse_age(r['f2_dob'], r['event_date']), axis=1)

    df['age_diff']    = df['f1_age']    - df['f2_age']
    df['reach_diff']  = df['f1_reach']  - df['f2_reach']
    df['height_diff'] = df['f1_height'] - df['f2_height']
    df['age_ratio']   = df['f1_age']    / df['f2_age']
    df['reach_ratio'] = df['f1_reach']  / df['f2_reach']
    df['height_ratio']= df['f1_height'] / df['f2_height']

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        ufc_ages = []
        for _, fight in df.iterrows():
            fid = fight[fid_col]
            if fid in fighter_fights_dict:
                first = fighter_fights_dict[fid].iloc[0]['event_date']
                days  = (datetime.strptime(fight['event_date'], "%Y-%m-%d") -
                         datetime.strptime(first,              "%Y-%m-%d")).days
                ufc_ages.append(days / 365.25)
            else:
                ufc_ages.append(0)
        df[f'{prefix}_ufc_age'] = ufc_ages

    df['diff_ufc_age'] = df['f1_ufc_age'] - df['f2_ufc_age']
    df = df.drop(['f1_dob', 'f2_dob'], axis=1)
    return df


fight_features = add_physical_and_experience(fight_features, conn)
print(f"✔ Shape with physical features: {fight_features.shape}")

✔ Shape with physical features: (2474, 309)


In [263]:
# ============================================================
# CELL 8b: Interaction Features
# ============================================================
# Motivated by disagreement analysis:
# - Correct upsets had diff_td_avg_dec_avg of +0.348 vs +0.083
# - Correct upsets had age_diff of +0.082 vs +0.910
# These interactions help the model learn that takedown and
# striking advantages mean MORE for younger fighters

# ── Age × Takedown ──────────────────────────────────────────
# Captures: young fighter with takedown advantage over older opponent
fight_features['age_td_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_td_avg_dec_avg']
)

# AdjPerf version — opponent-adjusted takedown advantage × age
fight_features['age_td_adjperf_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_td_avg_adjperf']
)

# ── Age × Striking ──────────────────────────────────────────
# Captures: young fighter with striking accuracy advantage
fight_features['age_str_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_str_acc_dec_avg']
)

# ── Age × Win Ratio ─────────────────────────────────────────
# Captures: younger fighter with better win record
fight_features['age_winratio_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_win_ratio']
)

# ── UFC Experience × Takedown ───────────────────────────────
# Captures: less experienced fighter compensating with grappling
fight_features['ufc_age_td_interaction'] = (
    fight_features['diff_ufc_age'] *
    fight_features['diff_td_avg_dec_avg']
)

# ── UFC Experience × Striking ───────────────────────────────
# Captures: less experienced fighter with striking edge
fight_features['ufc_age_str_interaction'] = (
    fight_features['diff_ufc_age'] *
    fight_features['diff_str_acc_dec_avg']
)

# ── Takedown × Control ──────────────────────────────────────
# Captures: fighters who take down AND control — complete grapplers
fight_features['td_ctrl_interaction'] = (
    fight_features['diff_td_avg_dec_avg'] *
    fight_features['diff_ctrl_time_per_min_dec_avg']
)

# ── Striking × Head Defense ─────────────────────────────────
# Captures: fighters who land more AND absorb less — striking dominance
fight_features['str_headdef_interaction'] = (
    fight_features['diff_str_acc_dec_avg'] *
    fight_features['diff_head_allowed_dec_avg']
)

new_cols = [
    'age_td_interaction', 'age_td_adjperf_interaction',
    'age_str_interaction', 'age_winratio_interaction',
    'ufc_age_td_interaction', 'ufc_age_str_interaction',
    'td_ctrl_interaction', 'str_headdef_interaction'
]

# Fill any NaNs from multiplication
fight_features[new_cols] = fight_features[new_cols].fillna(0)

print(f"✓ Added {len(new_cols)} interaction features")
print(f"✓ fight_features shape: {fight_features.shape}")
for col in new_cols:
    print(f"  {col}: mean={fight_features[col].mean():.3f}, "
          f"std={fight_features[col].std():.3f}")

✓ Added 8 interaction features
✓ fight_features shape: (2474, 317)
  age_td_interaction: mean=-0.943, std=8.467
  age_td_adjperf_interaction: mean=-0.554, std=5.935
  age_str_interaction: mean=-0.032, std=0.304
  age_winratio_interaction: mean=-0.200, std=1.439
  ufc_age_td_interaction: mean=-0.674, std=7.658
  ufc_age_str_interaction: mean=-0.037, std=0.271
  td_ctrl_interaction: mean=0.289, std=0.627
  str_headdef_interaction: mean=-0.009, std=0.057


In [273]:
# ============================================================
# CELL 9: Optuna Hyperparameter Search + Feature Trimming
# ============================================================

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Step 1: Train/Val split ---
train = fight_features[fight_features['event_date'] < '2024-01-01']
val   = fight_features[fight_features['event_date'] >= '2024-01-01']

FEATURE_COLS = [c for c in fight_features.columns 
                if c.startswith('diff_')
                or c in ['age_diff', 'age_ratio', 'reach_diff',
                         'height_diff', 'reach_ratio', 'height_ratio']]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train['winner']
X_val   = val[FEATURE_COLS].fillna(0)
y_val   = val['winner']

print(f"✓ Train: {X_train.shape}, Val: {X_val.shape}")

# --- Step 2: Feature trimming (use fixed baseline config) ---
model_for_imp = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    min_child_weight=15, subsample=0.6, colsample_bytree=0.5,
    gamma=2.0, reg_alpha=1.0, reg_lambda=2.0, random_state=42
)
model_for_imp.fit(X_train, y_train)

feat_imp = pd.Series(
    model_for_imp.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

threshold = 0.007
low_imp = feat_imp[feat_imp < threshold].index.tolist()
X_train_trimmed = X_train.drop(columns=low_imp)
X_val_trimmed   = X_val.drop(columns=low_imp)
print(f"✓ Trimmed to {X_train_trimmed.shape[1]} features (dropped {len(low_imp)})")

# --- Step 3: Optuna search ---
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample':        trial.suggest_float('subsample', 0.4, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state':     42,
    }
    
    m = XGBClassifier(**params)
    m.fit(X_train_trimmed, y_train)
    
    val_acc   = m.score(X_val_trimmed, y_val)
    train_acc = m.score(X_train_trimmed, y_train)
    gap = train_acc - val_acc
    
    # Penalize large train/val gaps to avoid overfitting configs
    if gap > 0.12:
        return val_acc - (gap - 0.12) * 0.5
    
    return val_acc

print(f"\n✓ Starting Optuna search (200 trials)...")
start = time.time()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

print(f"Done in {time.time()-start:.1f}s")

# --- Step 4: Results ---
best = study.best_trial
print(f"\n{'='*55}")
print(f"BEST TRIAL: #{best.number}")
print(f"{'='*55}")
print(f"Validation accuracy: {best.value:.4f} ({best.value*100:.1f}%)")
print(f"\nBest parameters:")
for k, v in best.params.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# --- Step 5: Retrain with best params and report ---
best_model = XGBClassifier(**best.params, random_state=42)
best_model.fit(X_train_trimmed, y_train)

train_acc = best_model.score(X_train_trimmed, y_train)
val_acc   = best_model.score(X_val_trimmed, y_val)

print(f"\n{'='*55}")
print(f"FINAL RESULTS (best params)")
print(f"{'='*55}")
print(f"Training:   {train_acc*100:.1f}%")
print(f"Validation: {val_acc*100:.1f}%")
print(f"Gap:        {(train_acc-val_acc)*100:.1f}%")

# --- Step 6: Compare to baseline ---
print(f"\n{'='*55}")
print(f"COMPARISON TO BASELINE")
print(f"{'='*55}")
print(f"Baseline:  65.8% val, 74.4% train, 8.6% gap")
print(f"Optuna:    {val_acc*100:.1f}% val, {train_acc*100:.1f}% train, "
      f"{(train_acc-val_acc)*100:.1f}% gap")
print(f"Change:    {(val_acc-0.658)*100:+.1f}%")

# --- Step 7: Top 5 trials ---
print(f"\n{'='*55}")
print(f"TOP 5 TRIALS")
print(f"{'='*55}")
trials_sorted = sorted(study.trials, key=lambda t: t.value, reverse=True)
for t in trials_sorted[:5]:
    m = XGBClassifier(**t.params, random_state=42)
    m.fit(X_train_trimmed, y_train)
    ta = m.score(X_train_trimmed, y_train)
    print(f"  Trial {t.number:>3}: val={t.value:.4f} train={ta:.4f} "
          f"gap={ta-t.value:.4f}")

# Store for later use
model_trimmed = best_model

✓ Train: (1857, 101), Val: (617, 101)
✓ Trimmed to 93 features (dropped 8)

✓ Starting Optuna search (200 trials)...
Done in 119.9s

BEST TRIAL: #184
Validation accuracy: 0.6775 (67.7%)

Best parameters:
  n_estimators: 158
  max_depth: 4
  learning_rate: 0.0587
  min_child_weight: 47
  subsample: 0.4171
  colsample_bytree: 0.3222
  gamma: 0.5586
  reg_alpha: 4.0401
  reg_lambda: 3.1794

FINAL RESULTS (best params)
Training:   68.2%
Validation: 67.7%
Gap:        0.4%

COMPARISON TO BASELINE
Baseline:  65.8% val, 74.4% train, 8.6% gap
Optuna:    67.7% val, 68.2% train, 0.4% gap
Change:    +1.9%

TOP 5 TRIALS
  Trial 184: val=0.6775 train=0.6817 gap=0.0043
  Trial 156: val=0.6759 train=0.6634 gap=-0.0124
  Trial 168: val=0.6759 train=0.6559 gap=-0.0200
  Trial  71: val=0.6726 train=0.6721 gap=-0.0006
  Trial  83: val=0.6726 train=0.6683 gap=-0.0043


In [274]:
# ============================================================
# CELL 10: AutoGluon with Author's Configuration
# ============================================================

from autogluon.tabular import TabularPredictor
import time

# --- Step 1: Train/Val split ---
train = fight_features[fight_features['event_date'] < '2024-01-01']
val   = fight_features[fight_features['event_date'] >= '2024-01-01']

FEATURE_COLS = [c for c in fight_features.columns 
                if c.startswith('diff_')
                or c in ['age_diff', 'age_ratio', 'reach_diff',
                         'height_diff', 'reach_ratio', 'height_ratio']]

# --- Step 2: Feature trimming (same as before) ---
model_for_imp = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    min_child_weight=15, subsample=0.6, colsample_bytree=0.5,
    gamma=2.0, reg_alpha=1.0, reg_lambda=2.0, random_state=42
)
model_for_imp.fit(train[FEATURE_COLS].fillna(0), train['winner'])

feat_imp = pd.Series(
    model_for_imp.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)

threshold = 0.007
low_imp = feat_imp[feat_imp < threshold].index.tolist()
keep_cols = [c for c in FEATURE_COLS if c not in low_imp]
print(f"✓ Trimmed to {len(keep_cols)} features")

# --- Step 3: Prepare AutoGluon dataframes ---
train_ag = train[keep_cols].fillna(0).copy()
train_ag['winner'] = train['winner'].values

val_ag = val[keep_cols].fillna(0).copy()
val_ag['winner'] = val['winner'].values

print(f"✓ Train: {train_ag.shape}, Val: {val_ag.shape}")

# --- Step 4: AutoGluon with author's config ---
print(f"\n✓ Starting AutoGluon training...")
start = time.time()

predictor = TabularPredictor(
    label='winner',
    eval_metric='accuracy',
    verbosity=2
).fit(
    train_data=train_ag,
    tuning_data=val_ag,          # Use val as holdout for tuning
    time_limit=1200,             # 20 minutes
    presets='best_quality',      # More thorough than 'good_quality'
    num_stack_levels=2,          # Author uses 2-level stacking
    num_bag_folds=4,             # Author uses n_splits=4
    num_bag_sets=2,              # Author uses num_bag_sets=2
    use_bag_holdout=True,        # Author uses this
)

print(f"\nDone in {time.time()-start:.1f}s")

# --- Step 5: Evaluate ---
train_acc = predictor.evaluate(train_ag)['accuracy']
val_acc   = predictor.evaluate(val_ag)['accuracy']

print(f"\n{'='*55}")
print(f"AUTOGLUON RESULTS")
print(f"{'='*55}")
print(f"Training:    {train_acc:.1%}")
print(f"Validation:  {val_acc:.1%}")
print(f"Gap:         {(train_acc - val_acc):.1%}")

print(f"\n{'='*55}")
print(f"COMPARISON")
print(f"{'='*55}")
print(f"Old baseline (XGB):    65.8% val, 74.4% train")
print(f"Optuna XGB:            67.7% val, 70.2% train")
print(f"AutoGluon stacked:     {val_acc:.1%} val, {train_acc:.1%} train")

# --- Step 6: Leaderboard ---
print(f"\n{'='*55}")
print(f"MODEL LEADERBOARD")
print(f"{'='*55}")
lb = predictor.leaderboard(val_ag)
print(lb[['model', 'score_val', 'score_test', 'fit_time']].head(15).to_string(index=False))

No path specified. Models will be saved in: "AutogluonModels\ag-20260220_190852"
Verbosity: 2 (Standard Logging)


✓ Trimmed to 93 features
✓ Train: (1857, 94), Val: (617, 94)

✓ Starting AutoGluon training...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.5
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          12
Pytorch Version:    2.10.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       2.63 GB / 15.94 GB (16.5%)
Disk Space Avail:   996.55 GB / 1675.37 GB (59.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=4, num_bag_sets=2
Beginning AutoGluon training ... Time limit = 1200s
AutoGluon will save models to "C:\Users\Sarthak\Documents\ML\fighter-beta\mma\notebooks\AutogluonModels\ag-20260220_190852"
Train Data Rows:    1857
Train Data Columns: 93
Tuning Data Rows:    617
Tuning Data Columns: 93
Label Column:       winner

[1000]	valid_set's binary_error: 0.403017


	0.6224	 = Validation score   (accuracy)
	47.07s	 = Training   runtime
	0.28s	 = Validation runtime
Fitting model: CatBoost_r177_BAG_L1 ... Training model for up to 379.79s of the 1046.51s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.624	 = Validation score   (accuracy)
	26.83s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: NeuralNetTorch_r79_BAG_L1 ... Training model for up to 352.70s of the 1019.42s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.6272	 = Validation score   (accuracy)
	22.07s	 = Training   runtime
	0.13s	 = Validation runtime
Fitting model: LightGBM_r131_BAG_L1 ... Training model for up to 330.24s of the 996.96s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.6207	 = Validation 

[1000]	valid_set's binary_error: 0.372845


	0.6305	 = Validation score   (accuracy)
	48.37s	 = Training   runtime
	0.21s	 = Validation runtime
Fitting model: CatBoost_r177_BAG_L2 ... Training model for up to 267.70s of the 489.65s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.6467	 = Validation score   (accuracy)
	26.92s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: NeuralNetTorch_r79_BAG_L2 ... Training model for up to 240.52s of the 462.47s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.6078	 = Validation score   (accuracy)
	45.5s	 = Training   runtime
	0.14s	 = Validation runtime
Fitting model: LightGBM_r131_BAG_L2 ... Training model for up to 194.60s of the 416.55s of remaining time.
	Fitting 8 child models (S1F1 - S2F4) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=6, gpus=0)
	0.6288	 = Validation sc


Done in 1199.1s

AUTOGLUON RESULTS
Training:    86.1%
Validation:  65.6%
Gap:         20.5%

COMPARISON
Old baseline (XGB):    65.8% val, 74.4% train
Optuna XGB:            67.7% val, 70.2% train
AutoGluon stacked:     65.6% val, 86.1% train

MODEL LEADERBOARD
                    model  score_val  score_test   fit_time
      WeightedEnsemble_L4   0.656402    0.656402 405.478306
      WeightedEnsemble_L3   0.656402    0.656402 433.047275
      WeightedEnsemble_L2   0.653160    0.653160  34.316753
        LightGBMXT_BAG_L2   0.653160    0.653160 405.414705
  RandomForestEntr_BAG_L1   0.649919    0.649919   0.881710
NeuralNetTorch_r22_BAG_L1   0.646677    0.646677  26.234508
     CatBoost_r177_BAG_L2   0.646677    0.646677 421.234359
      LightGBM_r96_BAG_L1   0.643436    0.643436   7.138991
    ExtraTrees_r42_BAG_L1   0.641815    0.641815   0.771121
          CatBoost_BAG_L2   0.641815    0.641815 427.498264
          CatBoost_BAG_L3   0.640194    0.640194 577.725795
    ExtraTreesEntr

In [266]:
import pickle

# Save model
pickle.dump(model_trimmed, open('xgb_model_v3.pkl', 'wb'))

# Save feature data with odds
fight_features.to_parquet('fight_features_v3.parquet')

# Save the trimmed feature column list
pickle.dump(X_val_trimmed.columns.tolist(), open('feature_cols_v3.pkl', 'wb'))

print("✔ Saved model, features, and column list")

✔ Saved model, features, and column list
